# Housing Price Prediction Using Machine Learning

Regression modeling, evaluation, and interpretation of residential property prices.

## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)


## 2. Load the Dataset


In [ ]:
from pathlib import Path
file_path = Path('data') / 'Housing_Price_Dataset.xlsx'

df = pd.read_excel(file_path, header=1)

print(f"Dataset shape: {df.shape}")
df.head()


## 3. Inspect the Data


In [ ]:
print(df.info())
df.head()


In [ ]:
# Missing values
missing_values = df.isnull().sum()
missing_values[missing_values > 0]


In [ ]:
# Descriptive statistics for numerical variables
df[['Area_SqFt', 'Rooms', 'Build_Year', 'Price']].describe()


## 4. Exploratory Data Analysis


### Chart 1: Distribution of Housing Prices


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['Price'].dropna(), bins=30, edgecolor='black', alpha=0.85)
plt.title('Distribution of Housing Prices')
plt.xlabel('Price')
plt.ylabel('Number of Properties')
plt.tight_layout()
plt.show()

print(f"Price skewness: {df['Price'].skew():.2f}")


### Chart 2: Correlation Heatmap


In [ ]:
corr = df[['Area_SqFt', 'Rooms', 'Build_Year', 'Price']].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)

variables = corr.columns.tolist()
ax.set_xticks(np.arange(len(variables)))
ax.set_yticks(np.arange(len(variables)))
ax.set_xticklabels(variables, rotation=45, ha='right')
ax.set_yticklabels(variables)

# Add correlation values to each square.
for i in range(len(variables)):
    for j in range(len(variables)):
        value = corr.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Correlation')
ax.set_title('Correlation Heatmap: Numerical Variables')
plt.tight_layout()
plt.show()


### Chart 3: Area vs. Housing Price


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Area_SqFt'], df['Price'], alpha=0.60)
plt.title('Area vs. Housing Price')
plt.xlabel('Area (Sq. Ft.)')
plt.ylabel('Price')
plt.tight_layout()
plt.show()


### Chart 4: Number of Rooms vs. Housing Price


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Rooms'], df['Price'], alpha=0.55)
plt.title('Number of Rooms vs. Housing Price')
plt.xlabel('Rooms')
plt.ylabel('Price')
plt.xticks(sorted(df['Rooms'].dropna().unique()))
plt.tight_layout()
plt.show()


## 5. Prepare the Data


In [ ]:
X = df.drop(columns=['Price'])
y = df['Price']

numeric_features = ['Area_SqFt', 'Rooms', 'Build_Year']
categorical_features = [
    'Location',
    'Street_Type',
    'Furnishing',
    'Property_Type',
    'Has_Pool'
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


## 6. Build and Evaluate Models


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regression': DecisionTreeRegressor(
        max_depth=8,
        min_samples_leaf=5,
        random_state=42
    ),
    'Random Forest Regression': RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    cv_rmse = np.sqrt(-cv_scores)

    results.append([
        name,
        rmse,
        mae,
        r2,
        cv_rmse.mean(),
        cv_rmse.std()
    ])

    fitted_models[name] = (pipeline, predictions)

results_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'Test RMSE',
        'Test MAE',
        'R-squared',
        '5-Fold CV RMSE',
        'CV SD'
    ]
)

results_df.round(3)


## 7. Compare Model Performance


### Chart 5: Model Error Comparison


In [ ]:
plot_df = results_df.set_index('Model')[['Test RMSE', 'Test MAE']]

ax = plot_df.plot(kind='bar', figsize=(9, 5))
plt.title('Model Error Comparison on Test Data')
plt.xlabel('Model')
plt.ylabel('Prediction Error')
plt.xticks(rotation=0)
plt.legend(['RMSE', 'MAE'])
plt.tight_layout()
plt.show()


## 8. Linear Regression Diagnostics


### Chart 6: Actual vs. Predicted Price


In [ ]:
linear_model, linear_predictions = fitted_models['Linear Regression']

plt.figure(figsize=(7, 6))
plt.scatter(y_test, linear_predictions, alpha=0.65)

low = min(y_test.min(), linear_predictions.min())
high = max(y_test.max(), linear_predictions.max())

plt.plot(
    [low, high],
    [low, high],
    linestyle='--',
    label='Perfect Prediction'
)

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Linear Regression: Actual vs. Predicted Price')
plt.legend()
plt.tight_layout()
plt.show()


### Chart 7: Residual Plot


In [ ]:
residuals = y_test - linear_predictions

plt.figure(figsize=(7, 5))
plt.scatter(linear_predictions, residuals, alpha=0.65)
plt.axhline(0, linestyle='--')

plt.xlabel('Predicted Price')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Linear Regression Residual Plot')
plt.tight_layout()
plt.show()


## 9. Interpret Linear Regression Predictors


In [ ]:
feature_names = (
    linear_model
    .named_steps['preprocessor']
    .get_feature_names_out()
)

coefficients = linear_model.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Clean the pipeline prefixes so labels are easier to read.
coef_df['Feature'] = (
    coef_df['Feature']
    .str.replace('num__', '', regex=False)
    .str.replace('cat__', '', regex=False)
)

coef_df['Absolute_Coefficient'] = coef_df['Coefficient'].abs()

coef_df.sort_values(
    'Absolute_Coefficient',
    ascending=False
).head(15)


### Chart 8: Largest Linear Regression Effects


In [ ]:
top_effects = (
    coef_df
    .sort_values('Absolute_Coefficient', ascending=False)
    .head(10)
    .sort_values('Coefficient')
)

plt.figure(figsize=(9, 6))
plt.barh(top_effects['Feature'], top_effects['Coefficient'])
plt.axvline(0, linewidth=1)

plt.title('Largest Linear Regression Effects')
plt.xlabel('Linear Regression Coefficient')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 10. Final Results


In [ ]:
final_results = results_df.sort_values('Test RMSE').copy()

final_results[
    ['Test RMSE', 'Test MAE', '5-Fold CV RMSE', 'CV SD']
] = final_results[
    ['Test RMSE', 'Test MAE', '5-Fold CV RMSE', 'CV SD']
].round(2)

final_results['R-squared'] = final_results['R-squared'].round(3)

final_results


### Model selection

Compare the holdout test metrics with cross-validation results before selecting a model. Linear Regression performed best in the original run; confirm the ranking after rerunning this cleaned notebook.